# Phase 3 (parts 1-2) - Gold layer on Databricks

Run top to bottom **after** `run_phase2_databricks.py`, because Gold reads
the Silver tables that notebook builds. It:

1. builds the Gold star schema from Silver
2. runs it a second time to prove the refresh is idempotent
3. shows the three Phase 1 business questions answered from Gold
4. shows the Silver-to-Gold rows in the same audit log as earlier layers
5. shows the read optimisation (liquid clustering / OPTIMIZE)
6. pushes a *new* snapshot through Bronze -> Silver -> Gold in one call

Screenshot sections 2, 3, 4, 5 and 6 for the report.

In [0]:
import os, sys, time

REPO = os.path.dirname(os.getcwd())          # this notebook lives in <repo>/notebooks
assert os.path.exists(f"{REPO}/pipelines"), f"pipelines/ not found under {REPO}"
if REPO not in sys.path:
    sys.path.insert(0, REPO)

VOLUME_NAME = "football_lakehouse"


def _first_writable_catalog_schema():
    prefer = ["workspace", "main"]
    cats = [r[0] for r in spark.sql("SHOW CATALOGS").collect()]  # noqa: F821
    ordered = [c for c in prefer if c in cats] + [c for c in cats if c not in prefer
                                                  and c not in ("samples", "system", "hive_metastore")]
    for cat in ordered:
        try:
            schemas = [r[0] for r in spark.sql(f"SHOW SCHEMAS IN `{cat}`").collect()]  # noqa: F821
        except Exception:
            continue
        for sch in (["default"] + [s for s in schemas if s != "default"]):
            if sch in schemas and sch != "information_schema":
                return cat, sch
    return None, None


cat, sch = _first_writable_catalog_schema()
LAKE = f"/Volumes/{cat}/{sch}/{VOLUME_NAME}"
os.environ["LAKE_ROOT"] = LAKE
os.environ["TABLE_FORMAT"] = "delta"

for m in [m for m in sys.modules if m == "pipelines" or m.startswith("pipelines.")]:
    del sys.modules[m]

from pipelines.common import config                      # noqa: E402
from pipelines.common.audit import new_batch_id          # noqa: E402
from pipelines.gold import silver_to_gold                # noqa: E402
from pipelines import run_pipeline                       # noqa: E402

RAW_URI = f"{LAKE}/raw"
print("lake:", LAKE)

lake: /Volumes/workspace/default/football_lakehouse


## 1. Build Gold from Silver

In [0]:
t0 = time.time()
silver_to_gold.run_all(spark, new_batch_id("gold_p1"), "FULL")  # noqa: F821
print(f"\nGold built in {time.time() - t0:.0f}s")

[dim_date] read=14001 written=1508 (overwrite)
[dim_competition] read=14001 inserted=13 updated=0
[dim_season] read=14001 inserted=9 updated=0
[dim_team] read=284 inserted=316 updated=0
[dim_player] read=600 inserted=550 updated=0
[fact_matches] read=14001 inserted=14001 updated=0
[fact_team_match] read=14001 inserted=28002 updated=0
[fact_top_scorers] read=600 inserted=600 updated=0
[agg_team_season_performance] read=28002 written=916 (overwrite)
[agg_home_away_split] read=28002 written=1796 (overwrite)
[agg_team_matchday_progress] read=28002 written=22284 (overwrite)
[validation] 35 checks passed
[optimize] fact_matches: LIQUID_CLUSTER_BY(competition_key, season_key)
[optimize] fact_team_match: LIQUID_CLUSTER_BY(team_key, season_key)
[optimize] fact_top_scorers: LIQUID_CLUSTER_BY(competition_key, snapshot_date_key)

Gold built in 168s


In [0]:
def gold_counts():
    out = {}
    for t in silver_to_gold.GOLD_TABLES:
        out[t] = spark.read.format("delta").load(config.table_path("gold", t)).count()  # noqa: F821
    return out


counts_1 = gold_counts()

## 2. Idempotency: refresh Gold again, nothing may move

Same Silver, new batch id. Facts and dimensions go through `MERGE`
(second pass reports `inserted=0`), aggregates are overwritten. Row counts
must be identical.

In [0]:
silver_to_gold.run_all(spark, new_batch_id("gold_p2"), "FULL", optimize=False)  # noqa: F821
counts_2 = gold_counts()

ok = True
print(f"{'table':32}{'pass1':>10}{'pass2':>10}   idempotent")
for t, a in counts_1.items():
    b = counts_2[t]
    ok &= (a == b)
    print(f"{t:32}{a:>10,}{b:>10,}   {'YES' if a == b else 'NO  <-- FAIL'}")
print("\nGOLD IDEMPOTENCY:", "PASS" if ok else "FAIL")

[dim_date] read=14001 written=1508 (overwrite)
[dim_competition] read=14001 inserted=0 updated=13
[dim_season] read=14001 inserted=0 updated=9
[dim_team] read=284 inserted=0 updated=316
[dim_player] read=600 inserted=0 updated=550
[fact_matches] read=14001 inserted=0 updated=14001
[fact_team_match] read=14001 inserted=0 updated=28002
[fact_top_scorers] read=600 inserted=0 updated=600
[agg_team_season_performance] read=28002 written=916 (overwrite)
[agg_home_away_split] read=28002 written=1796 (overwrite)
[agg_team_matchday_progress] read=28002 written=22284 (overwrite)
[validation] 35 checks passed
table                                pass1     pass2   idempotent
dim_date                             1,508     1,508   YES
dim_competition                         13        13   YES
dim_season                               9         9   YES
dim_team                               316       316   YES
dim_player                             550       550   YES
fact_matches                     

## 3. The business questions, answered from Gold

Plain joins from a fact/aggregate to its dimensions on integer keys: this
is the shape Power BI's relationships will mirror.

In [0]:
for t in silver_to_gold.GOLD_TABLES:
    spark.read.format("delta").load(config.table_path("gold", t)).createOrReplaceTempView(t)  # noqa: F821

### Q1. Which teams performed best, Premier League 2025/26?

In [0]:
spark.sql("""
SELECT a.season_rank AS pos, t.team_name, a.matches_played AS p, a.wins AS w, a.draws AS d,
       a.losses AS l, a.goals_for AS gf, a.goals_against AS ga, a.goal_difference AS gd, a.points AS pts
FROM agg_team_season_performance a
JOIN dim_team t USING (team_key)
JOIN dim_competition c USING (competition_key)
JOIN dim_season s USING (season_key)
WHERE c.competition_code = 'PL' AND s.season_label = '2025/26'
ORDER BY a.season_rank
LIMIT 10
""").display()  # noqa: F821

pos,team_name,p,w,d,l,gf,ga,gd,pts
1,Arsenal FC,38,26,7,5,71,27,44,85
2,Manchester City FC,38,23,9,6,77,35,42,78
3,Manchester United FC,38,20,11,7,69,50,19,71
4,Aston Villa FC,38,19,8,11,56,49,7,65
5,Liverpool FC,38,17,9,12,63,53,10,60
6,AFC Bournemouth,38,13,18,7,58,54,4,57
7,Sunderland AFC,38,14,12,12,42,48,-6,54
8,Brighton & Hove Albion FC,38,14,11,13,52,46,6,53
9,Brentford FC,38,14,11,13,55,52,3,53
10,Chelsea FC,38,14,10,14,58,52,6,52


### Q2. Do teams do better at home than away? (all league competitions)

In [0]:
spark.sql("""
SELECT a.venue,
       SUM(a.matches_played)                                       AS matches,
       ROUND(SUM(a.goals_for) / SUM(a.matches_played), 2)          AS avg_goals_for,
       ROUND(SUM(a.wins)   * 1.0 / SUM(a.matches_played), 3)       AS win_rate,
       ROUND(SUM(a.draws)  * 1.0 / SUM(a.matches_played), 3)       AS draw_rate,
       ROUND(SUM(a.points) * 1.0 / SUM(a.matches_played), 2)       AS points_per_game
FROM agg_home_away_split a
JOIN dim_competition c USING (competition_key)
WHERE c.competition_type = 'LEAGUE'
GROUP BY a.venue
""").display()  # noqa: F821

venue,matches,avg_goals_for,win_rate,draw_rate,points_per_game
HOME,10647,1.53,0.438,0.258,1.57
AWAY,10647,1.24,0.304,0.258,1.17


### Q3. Top scorers (latest snapshot) and the teams they concentrate in

In [0]:
spark.sql("""
SELECT c.competition_code, p.player_name, t.team_name, f.goals, f.assists, f.played_matches, f.goals_per_match
FROM fact_top_scorers f
JOIN dim_player p USING (player_key)
JOIN dim_team t USING (team_key)
JOIN dim_competition c USING (competition_key)
WHERE f.snapshot_date = (SELECT MAX(snapshot_date) FROM fact_top_scorers)
ORDER BY f.goals DESC
LIMIT 10
""").display()  # noqa: F821

competition_code,player_name,team_name,goals,assists,played_matches,goals_per_match
BSA,Kevin Viveros,CA Paranaense,18,3,26,0.69
BSA,Pedro,CR Flamengo,16,6,28,0.57
PD,Raphinha,FC Barcelona,12,2,7,1.71
BSA,Gabriel Barbosa,Santos FC,11,3,20,0.55
BSA,Carlos Vinícius,Grêmio FBPA,10,1,27,0.37
WC,Kylian Mbappé,France,10,4,8,1.25
BSA,John Kennedy,Fluminense FC,9,1,19,0.47
BSA,Luciano Juba,EC Bahia,9,1,25,0.36
BSA,Matheus Pereira,Cruzeiro EC,9,4,25,0.36
BSA,Danilo dos Santos de Oliveira,Botafogo FR,9,2,18,0.5


## 4. Execution log: Silver-to-Gold is in the same table as the other layers

For aggregates `rows_read` is the number of rows aggregated and
`rows_inserted` the number of groups produced; `duration_seconds` is the
execution time.

In [0]:
log = spark.read.format("delta").load(config.EXECUTION_LOG_PATH)  # noqa: F821
(log.filter("layer = 'Silver-to-Gold'")
    .select("batch_id", "table_name", "parameter", "load_type", "status",
            "rows_read", "rows_inserted", "rows_updated", "duration_seconds", "start_time")
    .orderBy("start_time", "table_name").display())

batch_id,table_name,parameter,load_type,status,rows_read,rows_inserted,rows_updated,duration_seconds,start_time
gold_p1_20261024T211305_fc4ccb,dim_date,OVERWRITE,FULL,SUCCESS,14001,1508,0,5,2026-10-24T21:13:05.750Z
gold_p1_20261024T211305_fc4ccb,dim_competition,MERGE,FULL,SUCCESS,14001,13,0,5,2026-10-24T21:13:12.715Z
gold_p1_20261024T211305_fc4ccb,dim_season,MERGE,FULL,SUCCESS,14001,9,0,4,2026-10-24T21:13:19.071Z
gold_p1_20261024T211305_fc4ccb,dim_team,MERGE,FULL,SUCCESS,284,316,0,6,2026-10-24T21:13:24.881Z
gold_p1_20261024T211305_fc4ccb,dim_player,MERGE,FULL,SUCCESS,600,550,0,4,2026-10-24T21:13:32.525Z
gold_p1_20261024T211305_fc4ccb,fact_matches,MERGE,FULL,SUCCESS,14001,14001,0,4,2026-10-24T21:13:38.074Z
gold_p1_20261024T211305_fc4ccb,fact_team_match,MERGE,FULL,SUCCESS,14001,28002,0,4,2026-10-24T21:13:43.780Z
gold_p1_20261024T211305_fc4ccb,fact_top_scorers,MERGE,FULL,SUCCESS,600,600,0,4,2026-10-24T21:13:49.847Z
gold_p1_20261024T211305_fc4ccb,agg_team_season_performance,OVERWRITE,FULL,SUCCESS,28002,916,0,3,2026-10-24T21:13:55.089Z
gold_p1_20261024T211305_fc4ccb,agg_home_away_split,OVERWRITE,FULL,SUCCESS,28002,1796,0,3,2026-10-24T21:13:59.292Z


In [0]:
log.groupBy("layer", "status").count().orderBy("layer", "status").display()

layer,status,count
Bronze-to-Silver,SUCCESS,16
Gold-Optimize,SUCCESS,3
Gold-Validation,SUCCESS,70
Raw-to-Bronze,SUCCESS,17
Silver-to-Gold,SUCCESS,22


In [0]:
# the validation checks write their own rows: rows_read = rows tested,
# rows_quarantined = violations (grain duplicates or orphan foreign keys)
(log.filter("layer = 'Gold-Validation'")
    .select("table_name", "parameter", "status", "rows_read", "rows_quarantined")
    .orderBy("table_name", "parameter").display())

table_name,parameter,status,rows_read,rows_quarantined
agg_home_away_split,fk:competition_key->dim_competition.competition_key,SUCCESS,1796,0
agg_home_away_split,fk:competition_key->dim_competition.competition_key,SUCCESS,1796,0
agg_home_away_split,fk:season_key->dim_season.season_key,SUCCESS,1796,0
agg_home_away_split,fk:season_key->dim_season.season_key,SUCCESS,1796,0
agg_home_away_split,fk:team_key->dim_team.team_key,SUCCESS,1796,0
agg_home_away_split,fk:team_key->dim_team.team_key,SUCCESS,1796,0
agg_home_away_split,grain:team_key+competition_key+season_key+venue,SUCCESS,1796,0
agg_home_away_split,grain:team_key+competition_key+season_key+venue,SUCCESS,1796,0
agg_team_matchday_progress,fk:competition_key->dim_competition.competition_key,SUCCESS,22284,0
agg_team_matchday_progress,fk:competition_key->dim_competition.competition_key,SUCCESS,22284,0


## 5. Read optimisation

`optimize_gold()` enables liquid clustering on the columns the dashboard
filters and joins on, then runs `OPTIMIZE`. If a workspace rejects
clustering it falls back to Z-ORDER, then plain `OPTIMIZE`; the
`parameter` column below shows which one actually ran.

In [0]:
(log.filter("layer = 'Gold-Optimize'")
    .select("table_name", "parameter", "status", "duration_seconds", "error_message")
    .orderBy("start_time").display())

table_name,parameter,status,duration_seconds,error_message
fact_matches,"LIQUID_CLUSTER_BY(competition_key, season_key)",SUCCESS,11,null
fact_team_match,"LIQUID_CLUSTER_BY(team_key, season_key)",SUCCESS,7,null
fact_top_scorers,"LIQUID_CLUSTER_BY(competition_key, snapshot_date_key)",SUCCESS,7,null


In [0]:
for t in silver_to_gold.CLUSTER_PLAN:
    p = config.table_path("gold", t)
    print(f"--- {t}")
    spark.sql(f"DESCRIBE DETAIL delta.`{p}`").select(  # noqa: F821
        "numFiles", "sizeInBytes", "clusteringColumns", "partitionColumns").show(truncate=False)

--- fact_matches
+--------+-----------+-----------------------------+----------------+
|numFiles|sizeInBytes|clusteringColumns            |partitionColumns|
+--------+-----------+-----------------------------+----------------+
|1       |220233     |[competition_key, season_key]|[]              |
+--------+-----------+-----------------------------+----------------+

--- fact_team_match
+--------+-----------+----------------------+----------------+
|numFiles|sizeInBytes|clusteringColumns     |partitionColumns|
+--------+-----------+----------------------+----------------+
|1       |310427     |[team_key, season_key]|[]              |
+--------+-----------+----------------------+----------------+

--- fact_top_scorers
+--------+-----------+------------------------------------+----------------+
|numFiles|sizeInBytes|clusteringColumns                   |partitionColumns|
+--------+-----------+------------------------------------+----------------+
|1       |11242      |[competition_key, snap

In [0]:
p = config.table_path("gold", "fact_team_match")
spark.sql(f"DESCRIBE HISTORY delta.`{p}`").select(  # noqa: F821
    "version", "timestamp", "operation", "operationParameters").display()

version,timestamp,operation,operationParameters
5,2026-10-24T21:16:59.000Z,MERGE,"Map(predicate -> [""((match_id#83962L = match_id#83669L) AND (venue#83972 = venue#83735))""], clusterBy -> [""team_key"",""season_key""], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}], clusteringOnWriteStatus -> null)"
4,2026-10-24T21:15:41.000Z,OPTIMIZE,"Map(predicate -> [], auto -> false, clusterBy -> [""team_key"",""season_key""], isFull -> false, zOrderBy -> [], batchId -> -1)"
3,2026-10-24T21:15:38.000Z,CLUSTER BY,"Map(oldClusteringColumns -> , newClusteringColumns -> team_key,season_key)"
2,2026-10-24T21:15:37.001Z,ROW TRACKING BACKFILL,Map(batchId -> 0)
1,2026-10-24T21:15:37.000Z,UPGRADE PROTOCOL,"Map(newProtocol -> {""minReaderVersion"":3,""minWriterVersion"":7,""readerFeatures"":[""deletionVectors""],""writerFeatures"":[""deletionVectors"",""domainMetadata"",""rowTracking"",""invariants"",""appendOnly""]})"
0,2026-10-24T21:13:49.000Z,WRITE,"Map(mode -> Overwrite, statsOnLoad -> false, partitionBy -> [])"


## 6. A new incremental file reaches Gold with no extra step

Replays the scorer and standings files as if they were a new day's
snapshot, plus one match file, and calls the end-to-end runner once.
Nobody calls `silver_to_gold` here: it is the third stage of the run.

In [0]:
INC_DATE = "2026-09-29"
snaps_before = sorted(str(r[0]) for r in spark.read.format("delta").load(  # noqa: F821
    config.table_path("gold", "fact_top_scorers")).select("snapshot_date").distinct().collect())
matches_before = spark.read.format("delta").load(config.table_path("gold", "fact_matches")).count()  # noqa: F821
print("fact_top_scorers snapshots before:", snaps_before)

batch = run_pipeline.run_end_to_end(
    spark, "INCREMENTAL", RAW_URI, INC_DATE,  # noqa: F821
    base_dir=f"{RAW_URI}/football-data/full_load", matches_glob="matches_PL_2026.json")

snaps_after = sorted(str(r[0]) for r in spark.read.format("delta").load(  # noqa: F821
    config.table_path("gold", "fact_top_scorers")).select("snapshot_date").distinct().collect())
matches_after = spark.read.format("delta").load(config.table_path("gold", "fact_matches")).count()  # noqa: F821
print("fact_top_scorers snapshots after :", snaps_after)
print(f"fact_matches rows: {matches_before:,} -> {matches_after:,} (same matches re-sent: updated, not duplicated)")
print("NEW SNAPSHOT REACHED GOLD:", "PASS" if INC_DATE in snaps_after else "FAIL")
print("NO DUPLICATE MATCHES     :", "PASS" if matches_before == matches_after else "FAIL")

fact_top_scorers snapshots before: ['2026-09-27']
[bronze_fd_matches] read=380 inserted=0 updated=380 quarantined=0
[bronze_fd_standings] read=212 inserted=212 updated=0 quarantined=0
[bronze_fd_scorers] read=600 inserted=600 updated=0 quarantined=0
[bronze_fd_teams] read=284 inserted=0 updated=284 quarantined=0
[silver_matches] read=14001 inserted=0 updated=14001 quarantined=0
[silver_standings] read=212 inserted=212 updated=0 quarantined=0
[silver_scorers] read=600 inserted=600 updated=0 quarantined=0
[silver_teams] read=284 inserted=0 updated=284 quarantined=0
[dim_date] read=14001 written=1508 (overwrite)
[dim_competition] read=14001 inserted=0 updated=13
[dim_season] read=14001 inserted=0 updated=9
[dim_team] read=284 inserted=0 updated=316
[dim_player] read=1200 inserted=0 updated=550
[fact_matches] read=14001 inserted=0 updated=14001
[fact_team_match] read=14001 inserted=0 updated=28002
[fact_top_scorers] read=1200 inserted=600 updated=600
[agg_team_season_performance] read=2800

In [0]:
# one batch id, three layers
(spark.read.format("delta").load(config.EXECUTION_LOG_PATH)  # noqa: F821
      .filter(f"batch_id = '{batch}' AND layer IN ('Raw-to-Bronze','Bronze-to-Silver','Silver-to-Gold')")
      .select("layer", "table_name", "load_type", "status", "rows_read", "rows_inserted",
              "rows_updated", "duration_seconds")
      .orderBy("layer", "table_name").display())

layer,table_name,load_type,status,rows_read,rows_inserted,rows_updated,duration_seconds
Bronze-to-Silver,silver_matches,INCREMENTAL,SUCCESS,14001,0,14001,8
Bronze-to-Silver,silver_scorers,INCREMENTAL,SUCCESS,600,600,0,7
Bronze-to-Silver,silver_standings,INCREMENTAL,SUCCESS,212,212,0,6
Bronze-to-Silver,silver_teams,INCREMENTAL,SUCCESS,284,0,284,8
Raw-to-Bronze,bronze_fd_matches,INCREMENTAL,SUCCESS,380,0,380,9
Raw-to-Bronze,bronze_fd_scorers,INCREMENTAL,SUCCESS,600,600,0,7
Raw-to-Bronze,bronze_fd_standings,INCREMENTAL,SUCCESS,212,212,0,7
Raw-to-Bronze,bronze_fd_teams,INCREMENTAL,SUCCESS,284,0,284,8
Silver-to-Gold,agg_home_away_split,INCREMENTAL,SUCCESS,28002,1796,0,2
Silver-to-Gold,agg_team_matchday_progress,INCREMENTAL,SUCCESS,28002,22284,0,2
